In [1]:
import pandas as pd
import numpy as np
import re
from functools import reduce

In [2]:
columns = ['id','source','lang','comment_text','toxic']

### Train

In [3]:
data1 = pd.read_csv('../../data/translate/french/french.csv')
data2 = pd.read_csv('../../data/translate/spanish/spanish.csv')
data3 = pd.read_csv('../../data/translate/russian/russian.csv')
data4 = pd.read_csv('../../data/translate/turkish/turkish.csv')
data5 = pd.read_csv('../../data/translate/italian/italian.csv')
data6 = pd.read_csv('../../data/translate/portugese/portugese.csv')
data7 = pd.read_csv('../../data/translate/data_fr.csv')
data8 = pd.read_csv('../../data/translate/data_es.csv')
data9 = pd.read_csv('../../data/translate/data_it.csv')
data10 = pd.read_csv('../../data/translate/data_pt.csv')
data11 = pd.read_csv('../../data/translate/data_ru.csv')
data12 = pd.read_csv('../../data/translate/data_tr.csv')

/usr/local/lib/python3.6/dist-packages/IPython/core/interactiveshell.py:3063: DtypeWarning: Columns (0) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [4]:
data = data1.append(data2).append(data3).append(data4)
data = data.append(data5).append(data6).append(data7)
data = data.append(data8).append(data9).append(data10)
data = data.append(data11).append(data12)
data = data.drop_duplicates()
data = data.sample(frac=1.).reset_index(drop=True)

In [5]:
index = pd.read_csv('../../data/process/english/train_english.csv')[['id','source']]
index['id'] = index['id'].astype(str)
data['id'] = data['id'].astype(str)

In [6]:
data = index.merge(data, on=['id','source'])

In [7]:
len(data), data.comment_text.nunique() , data['toxic'].sum(), data['toxic'].mean()

(2898334, 2828212, 601093, 0.2073925917440847)

In [8]:
data['lang'].value_counts()

tr    485122
es    485122
fr    485103
pt    481034
ru    480991
it    480962
Name: lang, dtype: int64

In [9]:
data.head()

,id,source,lang,toxic,comment_text
0,0000997932d777bf,2020-train,fr,0,Explication\nPourquoi les modifications apport...
1,0000997932d777bf,2020-train,es,0,Explicación\n¿Por qué se revertieron las edici...
2,0000997932d777bf,2020-train,tr,0,açıklama\nHardcore Metallica Fan kullanıcı adı...
3,0000997932d777bf,2020-train,ru,0,"объяснение\nПочему изменения, сделанные под мо..."
4,0000997932d777bf,2020-train,it,0,Spiegazione\nPerché le modifiche apportate con...


In [10]:
data['source'].value_counts()

2019-train    1500258
2020-train    1341754
prev-test       56322
Name: source, dtype: int64

In [11]:
data.to_csv('../../data/process/foreign/train_foreign.csv', index=False)

### Valid

In [12]:
data = pd.read_csv('../../data/raw/valid_extra.csv')
orig = pd.read_csv('../../data/raw/validation.csv')[['comment_text']]
orig['original'] = 1
data = data.merge(orig, on='comment_text', how='left')
data['original'] = data['original'].fillna(0).astype(int)

In [13]:
data['source'] = '2020-valid'
data = data[columns + ['original']]

In [14]:
len(data), data['toxic'].sum(), data['toxic'].mean(), data['original'].mean()

(47950, 7380, 0.15391032325338894, 0.16684045881126172)

In [15]:
data.head()

,id,source,lang,comment_text,toxic,original
0,0,2020-valid,es,Este usuario ni siquiera llega al rango de ...,0,1
1,1,2020-valid,es,El texto de esta entrada parece estar cubierto...,0,0
2,2,2020-valid,es,Vale. Sólo expongo mi pasado. Todo tiempo pasa...,1,1
3,3,2020-valid,es,"Como subtítulo de este artículo, tengo dudas s...",0,0
4,4,2020-valid,es,Supongo que tomarás el Portekizi como un ejemp...,0,0


In [16]:
data.to_csv('../../data/process/foreign/valid_foreign.csv', index=False)

### Test

In [17]:
data = pd.read_csv('../../data/raw/test_extra.csv')
orig = pd.read_csv('../../data/raw/test.csv')[['content']]
orig = orig.rename(columns={'content': 'comment_text'})
orig['original'] = 1
data = data.merge(orig, on='comment_text', how='left')
data['original'] = data['original'].fillna(0).astype(int)

In [18]:
data['source'] = '2020-test'
data = data[columns[:4] + ['original']]

In [19]:
len(data), data['original'].mean()

(382367, 0.16692863139339953)

In [20]:
data.head()

,id,source,lang,comment_text,original
0,0,2020-test,es,"Como doceavo médico, un escritor de wiki agreg...",0
1,1,2020-test,es,"Es posible, pero hasta ahora no veo la necesid...",0
2,2,2020-test,es,"Entonces eres uno de los conservadores, que pr...",0
3,3,2020-test,es,"Desafortunadamente, no sucedió, pero había alg...",0
4,4,2020-test,es,: Imagen: Problema de origen en la imagen de S...,0


In [21]:
data.to_csv('../../data/process/foreign/test_foreign.csv', index=False)